# **01 - Análisis Exploratorio de Datos (EDA) y Selección de Variables**
## **Proyecto: Impacto de la Inteligencia Artificial Generativa en Estudiantes**

Este notebook aborda la primera etapa del ciclo de vida del proyecto:
1. **Comprensión del problema y de los datos**.
2. **Auditoría de calidad**: Detección de nulos y verificación de tipos de datos.
3. **Análisis Exploratorio de Datos (EDA)**: Visualizaciones y agrupaciones.
4. **Selección de variables**: Justificación estadística (correlación y ANOVA) y exportación a datos procesados.

In [ ]:
# ============================================================
# CONFIGURACIÓN GENERAL E IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import os
import sys
import pandas as pd
import numpy as np
from scipy import stats

# Agregar directorio raíz para importar el paquete src
sys.path.append(os.path.abspath('..'))

import src.plots as plots
import src.features as features

print('Entorno de trabajo y librerías inicializadas correctamente.')

### 1. Ingesta y Calidad de los Datos

Cargamos el dataset original `data/raw/ai_student_impact_dataset.csv`.

In [ ]:
# Carga del dataset desde raw
df = pd.read_csv('../data/raw/ai_student_impact_dataset.csv')

print(f'Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
print('\nPrimeros registros:')
df.head()

#### Auditoría de Tipos de Datos y Valores Faltantes

In [ ]:
print('Tipos de datos de las columnas:')
print(df.dtypes)

print('\nConteo de valores nulos por columna:')
print(df.isnull().sum())

### 2. Análisis Exploratorio de Datos (EDA)

#### 2.1 Estadísticas Descriptivas

In [ ]:
print('Variables Numéricas:')
display(df.describe().round(2))

print('\nConteo de Categorías para Variables de Interés:')
for col in ['Major_Category', 'Primary_Use_Case', 'Prompt_Engineering_Skill', 'Institutional_Policy']:
    print(f'\nDistribución de {col}:')
    print(df[col].value_counts())

#### 2.2 Visualización de Relaciones Numéricas contra `Post_Semester_GPA`

In [ ]:
# Usamos la función personalizada de plots para generar los gráficos de regresión simple
plots.plot_eda_regplots(df, filepath='../outputs/figures/eda_regplots.png')

#### 2.3 Visualización de Relaciones Categóricas

In [ ]:
# Usamos la función personalizada para mostrar las distribuciones por categorías
plots.plot_eda_boxplots(df, filepath='../outputs/figures/eda_boxplots.png')

#### 2.4 Agrupamientos y Tabla Pivote

In [ ]:
print('--- Promedio de GPA por Habilidad de Prompting y Carrera ---')
display(df.groupby(['Major_Category', 'Prompt_Engineering_Skill'])['Post_Semester_GPA'].mean().unstack())

print('\n--- Tabla Pivote: Promedio de Horas Semanales de IA según Carrera y Caso de Uso ---')
plots.plot_pivot_heatmap(df, filepath='../outputs/figures/eda_pivot_heatmap.png')

### 3. Selección de Variables

#### 3.1 Análisis de Correlación de Pearson

In [ ]:
columnas_num = ['Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Traditional_Study_Hours', 
                'Anxiety_Level_During_Exams', 'Tool_Diversity', 'Perceived_AI_Dependency', 
                'Skill_Retention_Score', 'Post_Semester_GPA']

plots.plot_correlation_matrix(df, columnas_num, filepath='../outputs/figures/eda_correlation_matrix.png')

#### 3.2 Validación de Variables Categóricas usando ANOVA y Levene

In [ ]:
print('=== TEST DE LEVENE PARA HOMOGENEIDAD DE VARIANZAS (Major_Category) ===')
grupos_major = [df[df['Major_Category'] == m]['Post_Semester_GPA'] for m in df['Major_Category'].unique()]
lev_stat, lev_p = stats.levene(*grupos_major)
print(f'Estadístico: {lev_stat:.4f} | p-valor: {lev_p:.4e}')
if lev_p > 0.05:
    print('✔ Varianzas homogéneas (Supuesto cumplido)')
else:
    print('⚠ Varianzas NO homogéneas. ANOVA debe interpretarse con cautela.')

print('\n=== ANOVA GLOBAL (Major_Category vs Post_Semester_GPA) ===')
f_stat, p_val = stats.f_oneway(*grupos_major)
print(f'F-statistic: {f_stat:.4f} | p-valor: {p_val:.4e}')
if p_val < 0.05:
    print('✔ Existen diferencias significativas en el promedio de GPA según la carrera.')
else:
    print('✘ No hay diferencias significativas.')

### 4. Conclusión y Exportación

Exportamos el conjunto de variables seleccionadas como dataset procesado en `data/processed/`.

In [ ]:
# Usamos la función modularizada para seleccionar las variables
df_modelo = features.select_features(df)
df_modelo.to_csv('../data/processed/ai_student_impact_procesado.csv', index=False)
print('Dataset procesado exportado exitosamente a data/processed/ai_student_impact_procesado.csv')